In [ ]:
%pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 43.8 MB/s  0:00:02


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

In [ ]:
# Function to split data into 5 folds
# Returns X_train, X_test, y_train, y_test, and folds
def split_data(df):
    target_1 = ['2023-01-01', '2023-02-01', '2023-03-01', '2023-04-01', '2023-05-01', '2023-06-01', '2023-07-01', '2023-08-01']
    target_2 = ['2023-09-01', '2023-10-01', '2023-11-01', '2023-12-01', '2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01']
    target_3 = ['2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01']
    target_4 = ['2025-01-01', '2025-02-01', '2025-03-01', '2025-04-01', '2025-05-01', '2025-06-01', '2025-07-01', '2025-08-01']
    target_5 = ['2025-09-01', '2025-10-01', '2025-11-01', '2025-12-01', '2026-01-01', '2026-02-01', '2026-03-01']

    fold_1 = df[df['PERIOD_BEGIN'].isin(target_1)].copy()
    fold_2 = df[df['PERIOD_BEGIN'].isin(target_2)].copy()
    fold_3 = df[df['PERIOD_BEGIN'].isin(target_3)].copy()
    fold_4 = df[df['PERIOD_BEGIN'].isin(target_4)].copy()
    fold_5 = df[df['PERIOD_BEGIN'].isin(target_5)].copy()

    fold_1['fold_id'] = 0
    fold_2['fold_id'] = 1
    fold_3['fold_id'] = 2
    fold_4['fold_id'] = 3

    train = pd.concat([fold_1, fold_2, fold_3, fold_4], ignore_index=True)
    test = fold_5

    y_train = train.pop('AVG_SALE_TO_LIST')
    X_train = train.drop(columns=['PERIOD_BEGIN', 'PERIOD_END'])
    folds = X_train.pop('fold_id')
    y_test = test.pop('AVG_SALE_TO_LIST')
    X_test = test.drop(columns=['PERIOD_BEGIN', 'PERIOD_END'])

    return X_train, X_test, y_train, y_test, folds

In [ ]:
# Function to split data into 5 folds
# Returns X_train, X_test, y_train, y_test, and folds

#this function is slightly modified for the just demographic data file
def split_data_just_dem(df):
    target_1 = ['2023-01-01', '2023-02-01', '2023-03-01', '2023-04-01', '2023-05-01', '2023-06-01', '2023-07-01', '2023-08-01']
    target_2 = ['2023-09-01', '2023-10-01', '2023-11-01', '2023-12-01', '2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01']
    target_3 = ['2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01']
    target_4 = ['2025-01-01', '2025-02-01', '2025-03-01', '2025-04-01', '2025-05-01', '2025-06-01', '2025-07-01', '2025-08-01']
    target_5 = ['2025-09-01', '2025-10-01', '2025-11-01', '2025-12-01', '2026-01-01', '2026-02-01', '2026-03-01']

    fold_1 = df[df['PERIOD_BEGIN'].isin(target_1)].copy()
    fold_2 = df[df['PERIOD_BEGIN'].isin(target_2)].copy()
    fold_3 = df[df['PERIOD_BEGIN'].isin(target_3)].copy()
    fold_4 = df[df['PERIOD_BEGIN'].isin(target_4)].copy()
    fold_5 = df[df['PERIOD_BEGIN'].isin(target_5)].copy()

    fold_1['fold_id'] = 0
    fold_2['fold_id'] = 1
    fold_3['fold_id'] = 2
    fold_4['fold_id'] = 3

    train = pd.concat([fold_1, fold_2, fold_3, fold_4], ignore_index=True)
    test = fold_5

    y_train = train.pop('AVG_SALE_TO_LIST')
    #this is the first modification, which just drops the actual market data from the training set
    X_train = train.drop(columns=['PERIOD_BEGIN', 'PERIOD_END', 'MEDIAN_LIST_PRICE', 'MEDIAN_LIST_PPSF', 'HOMES_SOLD', 'PENDING_SALES', 'NEW_LISTINGS', 'INVENTORY', 'MEDIAN_DOM', 'OFF_MARKET_IN_TWO_WEEKS'])
    folds = X_train.pop('fold_id')
    y_test = test.pop('AVG_SALE_TO_LIST')
    #this is the second modification, which just drops the actual market data from the testing set
    X_test = test.drop(columns=['PERIOD_BEGIN', 'PERIOD_END', 'MEDIAN_LIST_PRICE', 'MEDIAN_LIST_PPSF', 'HOMES_SOLD', 'PENDING_SALES', 'NEW_LISTINGS', 'INVENTORY', 'MEDIAN_DOM', 'OFF_MARKET_IN_TWO_WEEKS'])

    return X_train, X_test, y_train, y_test, folds

In [ ]:
# Parameter tuning function
# Takes dataframe df, parameter grid param_grid

def parameter_tune(csv_filename, param_grid):
    # Get splits
    df = pd.read_csv(csv_filename)
    X_train, X_test, y_train, y_test, folds = split_data(df)

    # Get fold test/valid indices cuz CatBoost needs that
    folding_indices = []
    for fold_id in folds.unique():
        train_idx = np.where(folds != fold_id)[0]
        val_idx = np.where(folds == fold_id)[0]
        folding_indices.append((train_idx, val_idx))

    model = CatBoostRegressor(
        iterations=2000,
        loss_function='MAE',
        eval_metric='MAE',
        early_stopping_rounds=50,
        bootstrap_type='Bernoulli',    # Required to enable subsample
        max_ctr_complexity=1,
        random_seed=42,
        verbose=False
    )

    # Run grid search over folds
    result = model.grid_search(
        param_grid=param_grid,
        X=X_train,
        y=y_train,
        cv=folding_indices, # Overrides random split cuz we have it split manually
        search_by_train_test_split=False,
        plot=False
    )

    return result


In [ ]:
#This function is slightly modified of the above for the just demographic data file
def parameter_tune_just_dem(csv_filename, param_grid):
    # Get splits
    df = pd.read_csv(csv_filename)
    X_train, X_test, y_train, y_test, folds = split_data_just_dem(df)  #this is the only modification, just uses the modified data split function

    # Get fold test/valid indices cuz CatBoost needs that
    folding_indices = []
    for fold_id in folds.unique():
        train_idx = np.where(folds != fold_id)[0]
        val_idx = np.where(folds == fold_id)[0]
        folding_indices.append((train_idx, val_idx))

    model = CatBoostRegressor(
        iterations=2000,
        loss_function='MAE',
        eval_metric='MAE',
        early_stopping_rounds=50,
        bootstrap_type='Bernoulli',    # Required to enable subsample
        max_ctr_complexity=1,
        random_seed=42,
        verbose=False
    )

    # Run grid search over folds
    result = model.grid_search(
        param_grid=param_grid,
        X=X_train,
        y=y_train,
        cv=folding_indices, # Overrides random split cuz we have it split manually
        search_by_train_test_split=False,
        plot=False
    )

    return result


In [ ]:
param_grid = {
    'depth': [3, 4],
    'learning_rate': [0.03, 0.06, 0.09],
    'random_strength': [1, 5, 10],
    'subsample': [0.7, 0.85, 1.0],
    'rsm': [0.6, 0.8, 1.0],        # Subsamples columns so Redfin velocity doesn't drown out Census features
}

result = parameter_tune('/content/market_dem (2).csv', param_grid)
print(result)

Training on fold [0/4]

bestTest = 0.01313728963
bestIteration = 1861

Training on fold [1/4]

bestTest = 0.01214046855
bestIteration = 1204

Training on fold [2/4]

bestTest = 0.01082306934
bestIteration = 1833

Training on fold [3/4]

bestTest = 0.01052973685
bestIteration = 940

0:	loss: 0.0116609	best: 0.0116609 (0)	total: 12.5s	remaining: 33m 31s
Training on fold [0/4]

bestTest = 0.0131458106
bestIteration = 893

Training on fold [1/4]

bestTest = 0.0120128428
bestIteration = 927

Training on fold [2/4]

bestTest = 0.01089438713
bestIteration = 882

Training on fold [3/4]

bestTest = 0.01048126566
bestIteration = 746

1:	loss: 0.0116379	best: 0.0116379 (1)	total: 20.4s	remaining: 27m 14s
Training on fold [0/4]

bestTest = 0.01319489121
bestIteration = 566

Training on fold [1/4]

bestTest = 0.0120723451
bestIteration = 635

Training on fold [2/4]

bestTest = 0.01086820527
bestIteration = 686

Training on fold [3/4]

bestTest = 0.01060415522
bestIteration = 341

2:	loss: 0.0116902

In [ ]:
param_grid_2 = {
    'depth': [4],
    'learning_rate': [0.02, 0.03, 0.04],
    'random_strength': [3, 5, 7],
    'subsample': [0.5, 0.6, 0.7, 0.8],
    'rsm': [0.4, 0.5, 0.6, 0.7],        # Subsamples columns so Redfin velocity doesn't drown out Census features
}

result_2 = parameter_tune('/content/market_dem (2).csv', param_grid_2)
print(result_2)

Training on fold [0/4]

bestTest = 0.01318662977
bestIteration = 1646

Training on fold [1/4]

bestTest = 0.01203344194
bestIteration = 1382

Training on fold [2/4]

bestTest = 0.01084768199
bestIteration = 1711

Training on fold [3/4]

bestTest = 0.01043908636
bestIteration = 1461

0:	loss: 0.0116280	best: 0.0116280 (0)	total: 12.7s	remaining: 30m 14s
Training on fold [0/4]

bestTest = 0.01310290284
bestIteration = 1644

Training on fold [1/4]

bestTest = 0.0119928691
bestIteration = 1450

Training on fold [2/4]

bestTest = 0.01084201923
bestIteration = 1444

Training on fold [3/4]

bestTest = 0.01045617662
bestIteration = 998

1:	loss: 0.0116021	best: 0.0116021 (1)	total: 24.1s	remaining: 28m 34s
Training on fold [0/4]

bestTest = 0.01313161581
bestIteration = 963

Training on fold [1/4]

bestTest = 0.01198283406
bestIteration = 1058

Training on fold [2/4]

bestTest = 0.01087809922
bestIteration = 1244

Training on fold [3/4]

bestTest = 0.01048102125
bestIteration = 684

2:	loss: 0

In [ ]:
avg_best_iterations = (1327+1913+1321+1126)/4
print(avg_best_iterations)

1421.75


In [ ]:
param_grid_just_mark = {
    'depth': [3, 4],
    'learning_rate': [0.02, 0.03, 0.05],
    'random_strength': [5, 7, 9],
    'subsample': [0.5, 0.6, 0.7],
    'rsm': [0.5, 0.6, 0.7],        # Subsamples columns so Redfin velocity doesn't drown out Census features
}

result_just_dem = parameter_tune('/content/market (2).csv', param_grid_just_mark)
print(result_just_mark)

Training on fold [0/4]

bestTest = 0.01363488454
bestIteration = 1976

Training on fold [1/4]

bestTest = 0.01294047664
bestIteration = 1681

Training on fold [2/4]

bestTest = 0.01166371898
bestIteration = 1411

Training on fold [3/4]

bestTest = 0.01123479809
bestIteration = 1981

0:	loss: 0.0123701	best: 0.0123701 (0)	total: 9.25s	remaining: 24m 49s
Training on fold [0/4]

bestTest = 0.01363424948
bestIteration = 1277

Training on fold [1/4]

bestTest = 0.01294548657
bestIteration = 1123

Training on fold [2/4]

bestTest = 0.01167840418
bestIteration = 774

Training on fold [3/4]

bestTest = 0.01124181753
bestIteration = 1792

1:	loss: 0.0123776	best: 0.0123701 (0)	total: 15.5s	remaining: 20m 38s
Training on fold [0/4]

bestTest = 0.01358396863
bestIteration = 1120

Training on fold [1/4]

bestTest = 0.0129078665
bestIteration = 922

Training on fold [2/4]

bestTest = 0.01165953044
bestIteration = 607

Training on fold [3/4]

bestTest = 0.01130450337
bestIteration = 821

2:	loss: 0.

NameError: name 'result_just_mark' is not defined

In [ ]:
print(result_just_dem)

{'params': {'subsample': 0.5, 'rsm': 0.5, 'random_strength': 9, 'depth': 4, 'learning_rate': 0.02}, 'cv_results': defaultdict(<class 'list'>, {'iterations': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 19

In [ ]:
param_grid_just_mark_2 = {
    'depth': [4],
    'learning_rate': [0.01, 0.02, 0.03],
    'random_strength': [7, 9, 10],
    'subsample': [0.4, 0.5, 0.6],
    'rsm': [0.4, 0.5, 0.6],        # Subsamples columns so Redfin velocity doesn't drown out Census features
}

result_just_mark_2 = parameter_tune('/content/market (2).csv', param_grid_just_mark_2)
print(result_just_mark_2)

Training on fold [0/4]

bestTest = 0.0137162777
bestIteration = 1999

Training on fold [1/4]

bestTest = 0.01295014178
bestIteration = 1999

Training on fold [2/4]

bestTest = 0.01168714448
bestIteration = 1992

Training on fold [3/4]

bestTest = 0.01130354446
bestIteration = 1999

0:	loss: 0.0124143	best: 0.0124143 (0)	total: 11.1s	remaining: 14m 44s
Training on fold [0/4]

bestTest = 0.01352129168
bestIteration = 1999

Training on fold [1/4]

bestTest = 0.01290668506
bestIteration = 1492

Training on fold [2/4]

bestTest = 0.01169750418
bestIteration = 1049

Training on fold [3/4]

bestTest = 0.01125344962
bestIteration = 1265

1:	loss: 0.0123460	best: 0.0123460 (1)	total: 18.9s	remaining: 12m 27s
Training on fold [0/4]

bestTest = 0.01352647184
bestIteration = 1554

Training on fold [1/4]

bestTest = 0.01293445665
bestIteration = 996

Training on fold [2/4]

bestTest = 0.01170595543
bestIteration = 892

Training on fold [3/4]

bestTest = 0.01122615459
bestIteration = 1453

2:	loss: 

In [ ]:
avg_best_iter_2 = (1568+1253+746+1626)/4
print(avg_best_iter_2)

1298.25


In [ ]:
param_grid_just_dem_1 = {
    'depth': [3, 4],
    'learning_rate': [0.02, 0.03, 0.05],
    'random_strength': [5, 7, 9],
    'subsample': [0.5, 0.6, 0.7],
    'rsm': [0.5, 0.6, 0.7],        # Subsamples columns so Redfin velocity doesn't drown out Census features
}

result_just_dem_1 = parameter_tune_just_dem('/content/market_dem (2).csv', param_grid_just_dem_1)
print(result_just_dem_1)

Training on fold [0/4]

bestTest = 0.01875777869
bestIteration = 1999

Training on fold [1/4]

bestTest = 0.01492806578
bestIteration = 1998

Training on fold [2/4]

bestTest = 0.01424723054
bestIteration = 1998

Training on fold [3/4]

bestTest = 0.01359591933
bestIteration = 1998

0:	loss: 0.0153823	best: 0.0153823 (0)	total: 11.5s	remaining: 30m 50s
Training on fold [0/4]

bestTest = 0.01862739274
bestIteration = 1978

Training on fold [1/4]

bestTest = 0.01476167274
bestIteration = 1998

Training on fold [2/4]

bestTest = 0.01413061568
bestIteration = 1999

Training on fold [3/4]

bestTest = 0.01350621878
bestIteration = 1994

1:	loss: 0.0152571	best: 0.0152571 (1)	total: 23.1s	remaining: 30m 45s
Training on fold [0/4]

bestTest = 0.0185709407
bestIteration = 1761

Training on fold [1/4]

bestTest = 0.01461055477
bestIteration = 1997

Training on fold [2/4]

bestTest = 0.01410271878
bestIteration = 1571

Training on fold [3/4]

bestTest = 0.01356999696
bestIteration = 1021

2:	loss

In [ ]:
param_grid_just_dem_2 = {
    'depth': [4],
    'learning_rate': [0.04, 0.05, 0.06, 0.07],
    'random_strength': [3, 4, 5, 6],
    'subsample': [0.6, 0.7, 0.8],
    'rsm': [0.5, 0.6],        # Subsamples columns so Redfin velocity doesn't drown out Census features
}

result_just_dem_2 = parameter_tune_just_dem('/content/market_dem (2).csv', param_grid_just_dem_2)
print(result_just_dem_2)

Training on fold [0/4]

bestTest = 0.01857213891
bestIteration = 993

Training on fold [1/4]

bestTest = 0.0145443406
bestIteration = 1816

Training on fold [2/4]

bestTest = 0.01402723211
bestIteration = 1069

Training on fold [3/4]

bestTest = 0.01344662196
bestIteration = 1100

0:	loss: 0.0151510	best: 0.0151510 (0)	total: 9.85s	remaining: 15m 35s
Training on fold [0/4]

bestTest = 0.01858974487
bestIteration = 762

Training on fold [1/4]

bestTest = 0.01455086649
bestIteration = 1643

Training on fold [2/4]

bestTest = 0.01400601839
bestIteration = 826

Training on fold [3/4]

bestTest = 0.01344449786
bestIteration = 673

1:	loss: 0.0151527	best: 0.0151510 (0)	total: 17.7s	remaining: 13m 52s
Training on fold [0/4]

bestTest = 0.01854960011
bestIteration = 882

Training on fold [1/4]

bestTest = 0.01455662238
bestIteration = 974

Training on fold [2/4]

bestTest = 0.01401247952
bestIteration = 824

Training on fold [3/4]

bestTest = 0.01349205124
bestIteration = 578

2:	loss: 0.0151

In [ ]:
avg_best_iter_3 = (620+1039+717+638)/4
print(avg_best_iter_3)

753.5
